In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import json
import time
import xarray as xr
import scipy.signal as signal
import glob
import os
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, StratifiedKFold, permutation_test_score, GridSearchCV
from sklearn.metrics import classification_report, recall_score, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score,make_scorer,balanced_accuracy_score, f1_score

import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns

from matplotlib.ticker import MultipleLocator

In [2]:
cmap1 = colors.ListedColormap([[0,0,0,0],[0.03137254901960784,0.18823529411764706,0.4196078431372549,1]])
cmap2 = colors.ListedColormap([[0,0,0,0],[0.8,0,0,1]])
cmap3 = colors.ListedColormap([[0,0,0,0],[0,0,1,1]])
cmap4 = colors.ListedColormap([[0,0,0,0],[0.8,0.8,0,1]])
cmap5 = colors.ListedColormap([[0,0,0,0],[0.7,0.7,0.7,1]])
cmap6 = colors.ListedColormap([[0,0,0,0],[0.9,0.9,0.9,1]])


c1 = (0.03137254901960784,0.18823529411764706,0.4196078431372549,1)
c2 = (0.8,0,0,1)    
c3 = (0,0,1)
c4 = (0.8,0.8,0)
c5 = (0.7,0.7,0.7)
c6 = (0.9,0.9,0.9)

In [3]:
data_dir = '/home/lukeew/Local_work/data/01_SLW_xgb'

v6_reference_dir = data_dir + '/CHCH_v6_reference/'
v6_reference_mask_dir = v6_reference_dir + 'processed{}/'
v6_model_mask_dir = data_dir + '/CHCH_v6_model/{}/model_processed_{}/'

output_dir = data_dir + '/CHCH_v6_model/figures/panels_all/'

In [5]:
def plot(date='20210518'):

    infile_ref_v6 = v6_reference_mask_dir.format(date[:-2])+str(date)+'.nc'
    print('  <--- ' + infile_ref_v6)
    ds_ref_v6 = xr.open_dataset(infile_ref_v6)

    infile_mod_01 = v6_model_mask_dir.format('AG_XGBoost',date[:-2])+str(date)+'.nc'
    print('  <--- ' + infile_mod_01)
    ds_mod_01 = xr.open_dataset(infile_mod_01)

    infile_mod_02 = v6_model_mask_dir.format('model_003',date[:-2])+str(date)+'.nc'
    print('  <--- ' + infile_mod_02)
    ds_mod_02 = xr.open_dataset(infile_mod_02)

    time = ds_ref_v6.time
    altitude = ds_ref_v6.level.values

    backscatter = ds_ref_v6.backscatter
    temp = ds_ref_v6.temp.values
    cloud_mask = ds_ref_v6.cloud_mask

    v6_vdr = ds_ref_v6.vdr

    v6_wlwc_ref = ds_ref_v6.layer_wlwc
    v6_slwc_ref = ds_ref_v6.layer_slwc
    v6_mpc_ref = ds_ref_v6.layer_mix
    v6_ic_ref = ds_ref_v6.layer_ice
    v6_unknown_ref = ds_ref_v6.layer_undet

    v6_slwc_t_ref = v6_slwc_ref.max(dim='level')

    # AG_XGBoost:

    m001_class_pks = ds_mod_01.classified_peaks.astype(float)
    m001_class_pks = m001_class_pks.where(m001_class_pks>=0,np.nan )
    m001_pks_t = ~m001_class_pks.max(dim='level').isnull()  # True if peaks are present, false if no peaks present
    m001_slwc_mod = ds_mod_01.slwc
    m001_slwc_t_mod = ds_mod_01.slwc_t
    m001_other_mod = ds_mod_01.other


    # Model 003:
    m002_class_pks = ds_mod_02.classified_peaks.astype(float)
    m002_class_pks = m002_class_pks.where(m002_class_pks>=0,np.nan )
    m002_pks_t = ~m002_class_pks.max(dim='level').isnull()  # True if peaks are present, false if no peaks present
    m002_wlwc_mod = ds_mod_02.wlwc
    m002_slwc_mod = ds_mod_02.slwc
    m002_neither_mod = ds_mod_02.neither
    m002_slwc_t_mod = ds_mod_02.slwc_t

    ds_ref_v6.close()
    ds_mod_01.close()
    ds_mod_02.close()



    ### Plot figure

    gridspec_kw = {
        'height_ratios':[8,8,8,1,8,1,8,1],
        # 'hspace':0.2
        }

    fig, axes = plt.subplots(8,figsize=(15,25), gridspec_kw=gridspec_kw)
    (ax1, ax2, ax3, ax4, ax5, ax6, ax7,ax8) = axes

    ## Fig 1 - ALCF backscatter

    Im1=ax1.imshow(backscatter[:].T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        interpolation='None',
        cmap='RdYlBu_r',
        norm=colors.LogNorm(vmin=1e-8,vmax=1e-4),
        
    )

    cax1 = ax1.inset_axes([1.02,0,0.02,1],transform=ax1.transAxes)

    fig.colorbar(Im1,
            extend='neither',
            cax=cax1,
        ).set_label(label='Att. vol. \nbackscattering coef. \n( m$^{-1}$sr$^{-1}$)', size=15, labelpad=6)

    ax1.set_ylabel('Altitude (km)', fontsize=15)
    ax1.set_xticklabels([])
    ax1.set_xticks(np.arange(36,288,36))
    ax1.xaxis.set_minor_locator(MultipleLocator(12))
    cax1.tick_params(labelsize=13)

    CS=ax1.contour(temp[:].T,
                levels=[-60,-50,-40,-30,-20,-10,0],
                colors='k',
                alpha=0.6,
                linestyles='dashed',
                extent=(0, 288, 0, 15),
                )

    ax1.clabel(CS,inline=True,fontsize=12,fmt='%.0f')


    ## Fig 2 - VDR raw data
        
    Im2=ax2.imshow(v6_vdr[:].T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap='RdYlBu'
    )

    ax2.set_xticklabels([])
    ax2.set_xticks(np.arange(36,288,36))
    ax2.xaxis.set_minor_locator(MultipleLocator(12))
    ax2.set_ylabel('Altitude (km)', fontsize=15)

    cax2 = ax2.inset_axes([1.02,0,0.02,1],transform=ax2.transAxes)

    fig.colorbar(Im2,
            extend='neither',
            cax=cax2,
        ).set_label(label='VDR', size=15, labelpad=6)

    cax2.tick_params(labelsize=13)

    ## Fig 3 - Reference (VDR) mask

    Im3=ax3.imshow(v6_ic_ref.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap4
    )

    Im3=ax3.imshow(v6_mpc_ref.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap1
    )

    Im3=ax3.imshow(v6_slwc_ref.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap2
    )

    Im3=ax3.imshow(v6_wlwc_ref.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap3
    )

    Im3=ax3.imshow(v6_unknown_ref.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap5
    )

    ax3.set_xticklabels([])
    ax3.set_xticks(np.arange(36,288,36))
    ax3.xaxis.set_minor_locator(MultipleLocator(12))
    ax3.set_ylabel('Altitude (km)', fontsize=15)

    ax3.contour(temp[:].T,
                levels=[-40,0],
                colors='k',
                linestyles=['dashed','solid'],
                extent=(0, 288, 0, 15),
                )


    colour_labels = ['SLWC', 'WLWC', 'MPC','IC', 'Undetermined']
    patch_colors = [ c2, c3, c1, c4, c5]
    patches_colours = [ mpatches.Patch(color=patch_colors[i], label=colour_labels[i] ) for i in range(5) ]

    patch_linestyles = ['dashed', 'solid']
    linestyle_labels = ['-40 \N{DEGREE SIGN}C', '0 \N{DEGREE SIGN}C']
    patches_lines = [mlines.Line2D([],[],color='k',linestyle=patch_linestyles[i], label=linestyle_labels[i]) for i in range(2)] 
    patches = patches_colours + patches_lines
    ax3.legend(handles=patches, bbox_to_anchor=(1.025, 0.3), loc=6, fontsize=13)#,borderaxespad=0. )


    ## Fig 4 - Reference mask profile view

    Im4=ax4.imshow([v6_slwc_t_ref],
        extent=(0, 288, 0, 1),
        aspect='auto',
        origin='lower',
        # vmin=0,
        # vmax=1,
        interpolation='None',
        cmap=cmap2
    )

    ax4.set_xticklabels([])
    ax4.set_yticklabels([])
    ax4.set_xticks(np.arange(36,288,36))
    ax4.set_yticks([])
    ax4.xaxis.set_minor_locator(MultipleLocator(12))    

    colour_labels = ['SLWC']
    patch_colors = [ c2]
    patches = [ mpatches.Patch(color=patch_colors[i], label=colour_labels[i] ) for i in range(1) ]

    ax4.legend(handles=patches, bbox_to_anchor=(1.025, 0.5), loc=6, fontsize=13)


    ## Fig 5 - Model 01 - G22 Davis

    Im5=ax5.imshow(cloud_mask.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap6
    )


    Im5=ax5.imshow(m001_other_mod.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap5
    )

    Im5=ax5.imshow(m001_slwc_mod.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap2
    )



    ax5.set_xticklabels([])
    ax5.set_xticks(np.arange(36,288,36))
    ax5.xaxis.set_minor_locator(MultipleLocator(12))
    ax5.set_ylabel('Altitude (km)', fontsize=15)

    colour_labels = ['SLCC', 'Non-SLCC Peak', 'Unknown']
    patch_colors = [ c2, c5, c6]
    patches_two = [ mpatches.Patch(color=patch_colors[i], label=colour_labels[i] ) for i in range(3) ]

    ax5.legend(handles=patches_two, bbox_to_anchor=(1.025, 0.25), loc=6, fontsize=13)#,borderaxespad=0. )


        
    ## Fig 6 - G22 Davis profile view

    Im6=ax6.imshow([m001_pks_t.values.astype(int)],
        extent=(0, 288, 0, 1),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap5
    )

    Im6=ax6.imshow([m001_slwc_t_mod],
        extent=(0, 288, 0, 1),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap2
    )

    colour_labels = ['SLCC', 'Non-SLCC']
    patch_colors = [ c2, c5]
    patches_one = [ mpatches.Patch(color=patch_colors[i], label=colour_labels[i] ) for i in range(2) ]

    ax6.legend(handles=patches_one, bbox_to_anchor=(1.025, 0.5), loc=6, fontsize=13)

    ax6.set_xticklabels([])
    ax6.set_yticklabels([])
    ax6.set_xticks(np.arange(36,288,36))
    ax6.set_yticks([])
    ax6.xaxis.set_minor_locator(MultipleLocator(12))   


    ## Fig 7 - G22 Christchurch

    Im7=ax7.imshow(cloud_mask.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap6
    )

    Im7=ax7.imshow(m002_neither_mod[:].T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap4
    )

    Im7=ax7.imshow(m002_slwc_mod.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap2
    )

    Im7=ax7.imshow(m002_wlwc_mod.T,
        extent=(0, 288, 0, 15),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap3
    )


    ax7.set_xticklabels([])
    ax7.set_xticks(np.arange(36,288,36))
    ax7.xaxis.set_minor_locator(MultipleLocator(12))
    ax7.set_ylabel('Altitude (km)', fontsize=15)

    colour_labels = ['SLWC', 'WLWC','IC/MPC', 'Unknown']
    patch_colors = [ c2, c3, c4, c6]
    patches_two = [ mpatches.Patch(color=patch_colors[i], label=colour_labels[i] ) for i in range(4) ]

    ax7.legend(handles=patches_two, bbox_to_anchor=(1.025, 0.25), loc=6, fontsize=13)#,borderaxespad=0. )


        
    ## Fig 8 - G22 Chch profile view

    Im8=ax8.imshow([m002_pks_t.values.astype(int)],
        extent=(0, 288, 0, 1),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap5
    )

    Im8=ax8.imshow([m002_slwc_t_mod],
        extent=(0, 288, 0, 1),
        aspect='auto',
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='None',
        cmap=cmap2
    )

    colour_labels = ['SLWC', 'Non-SLWC']
    patch_colors = [ c2, c5]
    patches_one = [ mpatches.Patch(color=patch_colors[i], label=colour_labels[i] ) for i in range(2) ]

    ax8.legend(handles=patches_one, bbox_to_anchor=(1.025, 0.5), loc=6, fontsize=13)

    ax8.set_xticklabels([])
    ax8.set_yticklabels([])
    ax8.set_xticks(np.arange(36,288,36))
    ax8.set_yticks([])
    ax8.xaxis.set_minor_locator(MultipleLocator(12))   


    ax8.set_xlabel('Time (UTC)', fontsize=20)

    ax8.set_xticklabels([ '03',
                            '06',
                            '09',
                            '12',
                            '15',
                            '18',
                            '21'], fontsize=15)




    ax1.xaxis.tick_top()
    ax1.set_xticklabels([ '03',
                            '06',
                            '09',
                            '12',
                            '15',
                            '18',
                            '21'], fontsize=15)



    ax3.set_title('VDR Reference Cloud Mask',fontsize=20,transform=ax3.transAxes,x=0.5,y=0.88)
    ax5.set_title('G22-Davis Cloud Mask',fontsize=20,transform=ax5.transAxes,x=0.5,y=0.88)
    ax7.set_title('G22-Christchurch Cloud Mask',fontsize=20,transform=ax7.transAxes,x=0.5,y=0.88)

    ddate = date[:4]+'-'+date[4:6]+'-'+date[6:]
    # ax1.text(0.5,1.2,'Z_'+TARGET+'   '+ddate,fontsize=30,transform=ax1.transAxes,ha='center')
    ax1.text(0.5,1.2,ddate,fontsize=30,transform=ax1.transAxes,ha='center')

    letters='abcdefgh'

    for i,ax in enumerate(axes):
        ax.tick_params(axis='y', labelsize=13)
        ax.text(-0.05,1,'({})'.format(letters[i]),fontsize=25,transform=ax.transAxes,ha='right',va='top',weight='semibold')
        


    plt.tight_layout()

    output_path = output_dir+str(date)+'.png'

    print('Saving to',output_path)
    # plt.savefig(output_path,dpi=300)
    # plt.close()
    plt.show()

In [ ]:
plot('20210518') # Fig 4
plot('20210605') # Fig 5